# Restaurant POS Sales Analysis


## Objective

Analyze restaurant point-of-sale data to understand revenue patterns, menu performance, server performance, and whether order details can predict high-revenue transactions.


## Setup

Reusable data cleaning, feature engineering, and model training code lives in the `src/` folder. The notebook focuses on the analysis workflow and business interpretation.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_cleaning import clean_pos_data, get_data_quality_summary, load_pos_data
from src.feature_engineering import add_high_revenue_target, add_time_features
from src.model import get_feature_importance, train_revenue_classifier

DATA_PATH = PROJECT_ROOT / "Data" / "steakhouse_pos_simulated_data.csv"


## Data Loading and Preparation


In [ ]:
raw_df = load_pos_data(DATA_PATH)

df = clean_pos_data(raw_df)
df = add_time_features(df)
df = add_high_revenue_target(df)

df.head()


In [ ]:
df.info()


## Data Quality Checks


In [ ]:
get_data_quality_summary(df)


In [ ]:
duplicate_rows = df.duplicated().sum()
duplicate_rows


## Key Business Metrics


In [ ]:
total_revenue = df["Revenue"].sum()
total_orders = len(df)
average_order_value = df["Revenue"].mean()
total_units_sold = df["Quantity"].sum()

pd.DataFrame(
    {
        "Metric": [
            "Total Revenue",
            "Total Orders",
            "Average Order Value",
            "Total Units Sold",
        ],
        "Value": [
            total_revenue,
            total_orders,
            average_order_value,
            total_units_sold,
        ],
    }
)


## Exploratory Data Analysis


In [ ]:
category_revenue = (
    df.groupby("Category")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Revenue", "size"),
        Quantity=("Quantity", "sum"),
        Average_Order_Value=("Revenue", "mean"),
    )
    .sort_values("Revenue", ascending=False)
)

category_revenue["Revenue_Share"] = category_revenue["Revenue"] / total_revenue
category_revenue


In [ ]:
ax = category_revenue.sort_values("Revenue")["Revenue"].plot(
    kind="barh",
    figsize=(8, 5),
    color="#2F6F73",
)
ax.set_title("Total Revenue by Category")
ax.set_xlabel("Revenue ($)")
ax.set_ylabel("Category")
plt.tight_layout()
plt.show()


In [ ]:
top_items = (
    df.groupby("Menu Item")
    .agg(
        Revenue=("Revenue", "sum"),
        Quantity=("Quantity", "sum"),
        Orders=("Revenue", "size"),
        Average_Order_Value=("Revenue", "mean"),
    )
    .sort_values("Revenue", ascending=False)
)

top_items["Revenue_Share"] = top_items["Revenue"] / total_revenue
top_items.head(10)


In [ ]:
ax = top_items.head(10).sort_values("Revenue")["Revenue"].plot(
    kind="barh",
    figsize=(9, 5),
    color="#7A5C58",
)
ax.set_title("Top 10 Menu Items by Revenue")
ax.set_xlabel("Revenue ($)")
ax.set_ylabel("Menu Item")
plt.tight_layout()
plt.show()


## Time-Based Sales Trends


In [ ]:
daily_revenue = df.groupby("Date")["Revenue"].sum().reset_index()
daily_revenue["Weekday"] = daily_revenue["Date"].dt.day_name()

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

weekday_average_revenue = (
    daily_revenue.groupby("Weekday")["Revenue"]
    .mean()
    .reindex(weekday_order)
    .sort_values(ascending=False)
)

weekday_average_revenue


In [ ]:
ax = weekday_average_revenue.reindex(weekday_order).plot(
    kind="bar",
    figsize=(8, 5),
    color="#C08457",
)
ax.set_title("Average Daily Revenue by Weekday")
ax.set_xlabel("Weekday")
ax.set_ylabel("Average Daily Revenue ($)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
hourly_revenue = (
    df.groupby("Hour")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Revenue", "size"),
        Average_Order_Value=("Revenue", "mean"),
    )
    .sort_values("Revenue", ascending=False)
)

hourly_revenue.head(10)


In [ ]:
ax = hourly_revenue.sort_index()["Revenue"].plot(
    kind="line",
    marker="o",
    figsize=(9, 5),
    color="#3D5A80",
)
ax.set_title("Revenue by Hour")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Revenue ($)")
plt.tight_layout()
plt.show()


## Server Performance


In [ ]:
server_performance = (
    df.groupby("Server Name")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Revenue", "size"),
        Quantity=("Quantity", "sum"),
        Average_Order_Value=("Revenue", "mean"),
    )
    .sort_values("Revenue", ascending=False)
)

server_performance["Revenue_Share"] = server_performance["Revenue"] / total_revenue
server_performance


In [ ]:
ax = server_performance.sort_values("Revenue")["Revenue"].plot(
    kind="barh",
    figsize=(8, 5),
    color="#577590",
)
ax.set_title("Server Revenue Performance")
ax.set_xlabel("Revenue ($)")
ax.set_ylabel("Server")
plt.tight_layout()
plt.show()


In [ ]:
server_category_mix = (
    df.groupby(["Server Name", "Category"])["Quantity"]
    .sum()
    .reset_index()
)

server_category_mix["Percentage"] = (
    server_category_mix["Quantity"]
    / server_category_mix.groupby("Server Name")["Quantity"].transform("sum")
    * 100
)

server_category_mix.round(2)


## Machine Learning Model


In [ ]:
model_results = train_revenue_classifier(df)

print(f"Model accuracy: {model_results['accuracy']:.2%}")
print(model_results["classification_report"])


In [ ]:
feature_importance = get_feature_importance(model_results["model"])
feature_importance.head(20)


In [ ]:
ax = feature_importance.head(10).sort_values("Importance").plot(
    x="Feature",
    y="Importance",
    kind="barh",
    figsize=(9, 5),
    legend=False,
    color="#8A6FDF",
)
ax.set_title("Top Model Feature Importances")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()


## Business Recommendations

- Prioritize premium entree promotion, especially steak items that account for the majority of revenue.
- Introduce dessert upselling strategies to improve performance in the lowest-revenue category.
- Staff peak dinner hours carefully, especially between 7 PM and 9 PM.
- Use Saturday demand patterns to guide inventory planning and scheduling.
- Study top-performing server behavior to identify successful upselling or service patterns.
- Use predictive modeling as a supporting tool for identifying high-value order patterns, not as the sole basis for staffing or performance decisions.


## Conclusion

The analysis identifies key revenue drivers across menu items, weekdays, hours, and servers. The machine learning model adds a predictive layer by estimating whether an order is likely to be high revenue based on order characteristics.
